# 05 · Control Flow: Routing, Loops, Parallelism, `Send` and `Command`

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama.

Control flow is where you decide **who is in charge**: fixed code, or the
model. LangGraph gives you a small set of tools for it, and knowing which to
reach for is most of the skill of designing graphs.

**You will learn**
- **LLM routers**: letting a model choose a branch, but only among branches you defined
- **`Command`**: a node that updates state *and* chooses the next node in one step
- **Guarded cycles**: loops that stop gracefully (the evaluator-optimizer pattern)
- **Static fan-out**: a fixed set of parallel LLM calls, joined afterwards
- **`Send`**: dynamic fan-out (map-reduce), where the number of branches is decided at runtime
- Limiting concurrency, and a table for choosing the right mechanism

**Prerequisites:** notebooks 01–04. *Optional background:* `00_CONCEPTS/03`
covers the same patterns in plain Python.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import operator, time
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send
from common import get_model

model = get_model()

## 1 · Why: the control-flow toolbox

| Mechanism | Who decides the path | Decided when | Use for |
|---|---|---|---|
| `add_edge(a, b)` | you | when you build the graph | fixed order ("tests before commit") |
| `add_conditional_edges` | a router function (code *or* LLM) | after a node runs | choosing among known branches |
| `Command(goto=...)` from a node | the node itself | inside the node | "update state and jump" in one step |
| a cycle + counter | a router checking a guard | each iteration | refine-until-good loops |
| several edges out of one node | you | build time | a fixed set of parallel branches |
| `Send(node, payload)` | a router, at runtime | after a node runs | a variable number of parallel branches |

Keep one principle from `00_CONCEPTS` in mind: **route with code when code can
decide; use the model only where judgement is needed.**

## 2 · An LLM router

A support desk. Classification needs judgement (free-text tickets), so an LLM
decides the branch. Three rules make LLM routing safe:

1. **Structured output with a `Literal`**, so the model can only name branches that exist.
2. **The criteria in the prompt**, because notebook 03 showed that constrained
   decoding alone doesn't make the model *read* your field descriptions.
3. **A safe default**: if the model isn't confident, route to a human queue.

In [ ]:
class Route(BaseModel):
    department: Literal['billing', 'technical', 'human']
    confidence: float = Field(ge=0, le=1)


ROUTER_PROMPT = """Route this support ticket.
- billing: charges, refunds, invoices, payments
- technical: bugs, crashes, errors, performance
- human: anything else, or unclear
Give your confidence from 0 to 1."""

router_llm = model.with_structured_output(Route, method='function_calling')


class Ticket(TypedDict):
    text: str
    department: str
    reply: str


def classify(state: Ticket) -> dict:
    r = router_llm.invoke([('system', ROUTER_PROMPT), ('user', state['text'])])
    dept = r.department if r.confidence >= 0.6 else 'human'      # ← the safe default is CODE
    print(f'  router: {r.department} ({r.confidence:.2f}) → {dept}')
    return {'department': dept}

def billing(state: Ticket):   return {'reply': model.invoke(f"As billing support, reply in 1 sentence to: {state['text']}").content}
def technical(state: Ticket): return {'reply': model.invoke(f"As a support engineer, give 1 troubleshooting step for: {state['text']}").content}
def human(state: Ticket):     return {'reply': 'Queued for a human agent.'}


b = StateGraph(Ticket)
for fn in (classify, billing, technical, human):
    b.add_node(fn.__name__, fn)
b.add_edge(START, 'classify')
b.add_conditional_edges('classify', lambda s: s['department'], ['billing', 'technical', 'human'])
for n in ('billing', 'technical', 'human'):
    b.add_edge(n, END)
desk = b.compile()

for text in ['My card was charged twice for the same order.',
             'The export button throws a 500 error.',
             'Do you sponsor local football teams?']:
    print(text)
    print('  reply :', desk.invoke({'text': text})['reply'][:120], '\n')

The LLM chose the branch, but **only among the three you drew**, and a plain
`if` on `confidence` decides when not to trust it. The graph stays predictable
even though a model is making the call.

## 3 · `Command`: update state and route from inside a node

With conditional edges, the decision lives in a separate router function. Often
the node that does the work already knows where to go next. `Command` lets it
say so directly, returning an **update** and a **destination** together:

```python
def node(state) -> Command[Literal['a', 'b']]:      # the annotation lists possible targets
    return Command(update={'key': value}, goto='a')
```

The return annotation `Command[Literal[...]]` tells LangGraph (and
`draw_ascii`) which nodes this one can jump to. No `add_conditional_edges` is needed.

In [ ]:
class Order(TypedDict):
    amount: float
    log: Annotated[list[str], operator.add]


def check_order(state: Order) -> Command[Literal['auto_approve', 'manager_review']]:
    if state['amount'] <= 500:
        return Command(update={'log': [f"{state['amount']} ≤ 500: auto"]}, goto='auto_approve')
    return Command(update={'log': [f"{state['amount']} > 500: needs review"]}, goto='manager_review')

def auto_approve(state: Order):   return {'log': ['approved automatically']}
def manager_review(state: Order): return {'log': ['sent to manager']}


b = StateGraph(Order)
b.add_node('check_order', check_order)
b.add_node('auto_approve', auto_approve)
b.add_node('manager_review', manager_review)
b.add_edge(START, 'check_order')
b.add_edge('auto_approve', END)
b.add_edge('manager_review', END)
orders = b.compile()

print(orders.get_graph().draw_ascii())
for amount in (120, 4200):
    print(amount, '→', orders.invoke({'amount': amount, 'log': []})['log'])

**Conditional edge or `Command`?**
- Use a **conditional edge** when the routing logic is separate from the work,
  or reused, or you want the graph structure visible in one place.
- Use **`Command`** when the node that computes something is naturally the one
  that knows where to go next. It also matters for multi-agent **handoffs**
  (notebook 06), where an agent passes control to another agent.

## 4 · A guarded cycle: evaluator-optimizer

A writer drafts, a reviewer judges, and failing drafts go back with feedback.
Two guards make the loop safe:

- **A hard cap** on revisions, checked by the router.
- **A graceful exit**: when the cap is hit, finish with the best draft *flagged*,
  rather than crashing with `GraphRecursionError`.

The reviewer follows the rule from `00_CONCEPTS/03`: **exact checks in code**
(word count, required words) and **judgement by the model** (tone), which only
runs if the exact checks pass.

In [ ]:
class Verdict(BaseModel):
    passed: bool
    feedback: str


class Draft(TypedDict):
    brief: str
    draft: str
    feedback: str
    revision: int
    status: str


MAX_REVISIONS = 3
reviewer_llm = model.with_structured_output(Verdict, method='function_calling')


def write(state: Draft) -> dict:
    prompt = (f"Write a one-sentence release note for: {state['brief']}. "
              "Use ONLY facts from this brief; never invent numbers. Reply with the sentence only.")
    if state.get('feedback'):
        prompt += f"\nPrevious draft: {state['draft']}\nFix this feedback: {state['feedback']}"
    return {'draft': model.invoke(prompt).content.strip(), 'revision': state.get('revision', 0) + 1}


def review(state: Draft) -> dict:
    text, problems = state['draft'], []
    if len(text.split()) > 25:
        problems.append(f'max 25 words (has {len(text.split())})')
    if 'battery' not in text.lower():
        problems.append('must mention the battery')
    if not problems:                                   # only ask the model once exact rules pass
        v = reviewer_llm.invoke(f"Brief: {state['brief']}\nRelease note: {text}\n\n"
                                'Pass it if it is professional, free of hype, and states ONLY facts '
                                'from the brief. Do not ask for details the brief does not contain.')
        if not v.passed:
            problems.append(v.feedback)
    print(f"  v{state['revision']}: {text[:90]}{'…' if len(text) > 90 else ''}")
    print(f"      {'PASS' if not problems else 'FAIL: ' + '; '.join(problems)}")
    return {'feedback': '; '.join(problems), 'status': 'approved' if not problems else 'rejected'}


def after_review(state: Draft) -> Literal['write', 'finish']:
    if state['status'] == 'approved' or state['revision'] >= MAX_REVISIONS:
        return 'finish'
    return 'write'


def finish(state: Draft) -> dict:
    if state['status'] != 'approved':
        return {'status': f"FLAGGED for human review after {state['revision']} revisions"}
    return {}


b = StateGraph(Draft)
for fn in (write, review, finish):
    b.add_node(fn.__name__, fn)
b.add_edge(START, 'write')
b.add_edge('write', 'review')
b.add_conditional_edges('review', after_review, ['write', 'finish'])
b.add_edge('finish', END)
editor = b.compile()

out = editor.invoke({'brief': 'Courier C2 robots get a new battery pack with 30% more range'})
print('\nstatus:', out['status'])

Run it a few times: sometimes it passes on the first draft, sometimes it takes
revisions, and sometimes it hits the cap and is **flagged**. All three are
correct outcomes.

**A lesson from building this demo.** An earlier version told the reviewer to be
"strict" and demand "specific" notes. The writer responded by **inventing
specifics** ("15,000 mAh battery") that were in no brief. An evaluator's criteria
shape what the generator produces: reward specificity the input can't support,
and you reward hallucination. Tell the reviewer what *not* to require, and tell
the writer to use only the facts given.

Notice that the guard is ordinary state (`revision`) checked
by an ordinary router, so the loop can never run away, and the caller always
gets a result with an honest status.

## 5 · Static fan-out: a fixed set of parallel LLM calls

When you know the branches in advance, draw them as several edges out of one
node. Here, three analysts look at the same idea **at the same time**, and a
join node combines their notes. A reducer (`operator.add`) collects the
parallel writes.

In [ ]:
class Analysis(TypedDict):
    idea: str
    notes: Annotated[list[str], operator.add]
    verdict: str


def analyst(role, question):
    def fn(state: Analysis) -> dict:
        answer = model.invoke(f"You are the {role}. {question} Idea: {state['idea']}. One sentence.").content
        return {'notes': [f'{role}: {answer}']}
    fn.__name__ = role.replace(' ', '_')
    return fn

ANALYSTS = [('market analyst', 'Who would pay for this, and why?'),
            ('engineer', 'What is the hardest technical problem?'),
            ('risk officer', 'What is the biggest legal or safety risk?')]


def decide(state: Analysis) -> dict:
    notes = '\n'.join(state['notes'])
    return {'verdict': model.invoke(f'Given these notes, go or no-go, in one sentence:\n{notes}').content}


b = StateGraph(Analysis)
b.add_node('decide', decide)
for role, question in ANALYSTS:
    fn = analyst(role, question)
    b.add_node(fn.__name__, fn)
    b.add_edge(START, fn.__name__)          # fan-out
    b.add_edge(fn.__name__, 'decide')       # fan-in (equal-length branches)
b.add_edge('decide', END)
panel = b.compile()

t0 = time.perf_counter()
out = panel.invoke({'idea': 'robots that deliver groceries up apartment stairs', 'notes': []})
print(f'{time.perf_counter() - t0:.1f}s')
for n in out['notes']:
    print(' •', n[:140])
print('verdict:', out['verdict'])

## 6 · `Send`: dynamic fan-out (map-reduce)

Static fan-out needs you to know the branches when you build the graph. Often
you don't: "research each subtopic" (how many?), "summarise each file changed"
(which files?). **`Send`** creates branches at runtime.

A router returns a **list of `Send(node_name, payload)`**. LangGraph runs one
copy of `node_name` per `Send`, **in parallel**, each receiving **its own
payload** instead of the whole state. Their outputs are merged back through the
reducers:

```
                         ┌─► Send('research', {'topic': t1}) ─┐
 START → plan ──router──►├─► Send('research', {'topic': t2}) ─┼─► write → END
       (model decides N) └─► Send('research', {'topic': tN}) ─┘
          "map"                     parallel workers              "reduce"
```

This is the **orchestrator-workers** pattern: the model plans the subtasks, and
the graph runs them.

In [ ]:
class Plan(BaseModel):
    topics: list[str] = Field(description='2 to 4 short, distinct research subtopics')


class Report(TypedDict):
    question: str
    topics: list[str]
    findings: Annotated[list[str], operator.add]     # every worker appends here
    report: str


class ResearchTask(TypedDict):                        # a worker's PRIVATE input
    question: str                                     # ← the context it needs
    topic: str


planner = model.with_structured_output(Plan, method='function_calling')


def plan(state: Report) -> dict:
    topics = planner.invoke(f"Split into 2-4 research subtopics: {state['question']}").topics
    print('planned:', topics)
    return {'topics': topics}


def fan_out(state: Report) -> list[Send]:
    return [Send('research', {'question': state['question'], 'topic': t})   # one branch per topic
            for t in state['topics']]


def research(task: ResearchTask) -> dict:
    start = time.perf_counter()
    text = model.invoke(f"We are answering: {task['question']}\n"
                        f"Write exactly two factual sentences on this aspect: {task['topic']}. "
                        "No preamble, no list.").content
    return {'findings': [f"## {task['topic']}  ({time.perf_counter() - start:.1f}s)\n{text}"]}


def write(state: Report) -> dict:
    body = '\n\n'.join(state['findings'])
    return {'report': model.invoke(f"Answer '{state['question']}' in 3 sentences using:\n{body}").content}


b = StateGraph(Report)
b.add_node('plan', plan)
b.add_node('research', research)
b.add_node('write', write)
b.add_edge(START, 'plan')
b.add_conditional_edges('plan', fan_out, ['research'])
b.add_edge('research', 'write')
b.add_edge('write', END)
researcher = b.compile()

t0 = time.perf_counter()
out = researcher.invoke({'question': 'Should a city allow sidewalk delivery robots?', 'findings': []})
print(f'total {time.perf_counter() - t0:.1f}s\n')
print('\n\n'.join(out['findings']))
print('\nREPORT:', out['report'])

Things to notice:

- **The worker's input is its own payload** (`ResearchTask`), not the whole
  state. That keeps workers simple and prompts small, **but "small" must not mean
  "missing context"**. The first version of this demo sent only `{'topic': t}`.
  Asked about "Safety concerns and regulations" with no idea *what* was being
  researched, a worker wrote about factory safety equipment, and another covered
  US unemployment statistics. Each worker needs the minimum context to do its
  job, here the original question.
- **Outputs merge through the reducer** on `findings`. Without `operator.add`
  you'd get `InvalidUpdateError` (notebook 02).
- **`write` runs once**, after all workers finish: every `Send` branch is part
  of the same superstep.

**Limiting concurrency.** Twenty topics means twenty simultaneous model calls,
which can hit rate limits or overload a local GPU. `max_concurrency` in the run
config caps how many nodes run at once:

In [ ]:
class Mini(TypedDict):
    items: list[str]
    done: Annotated[list[str], operator.add]

def slow_worker(task: dict) -> dict:
    time.sleep(1.0)
    return {'done': [task['item']]}

b = StateGraph(Mini)
b.add_node('start', lambda s: {})
b.add_node('slow_worker', slow_worker)
b.add_edge(START, 'start')
b.add_conditional_edges('start', lambda s: [Send('slow_worker', {'item': i}) for i in s['items']], ['slow_worker'])
mini = b.compile()

for limit in (None, 2):
    t0 = time.perf_counter()
    mini.invoke({'items': list('abcdef'), 'done': []}, config={'max_concurrency': limit} if limit else None)
    print(f'max_concurrency={limit}: 6 one-second workers took {time.perf_counter() - t0:.1f}s')

## 7 · Putting the toolbox together: which mechanism when?

| You need… | Use |
|---|---|
| Always B after A | `add_edge('a', 'b')` |
| A model to choose among known branches | `add_conditional_edges` + structured output with `Literal` |
| The working node to also pick the next step | `Command(update=..., goto=...)` |
| Repeat until good enough | a cycle + counter in state + a graceful `finish` node |
| A fixed set of parallel steps | several edges out of one node, reducer on shared keys |
| A variable number of parallel steps | `Send` from a router (map), reducer + a join node (reduce) |
| To protect rate limits or a GPU | `config={'max_concurrency': n}` |
| A hard safety stop | `config={'recursion_limit': n}` (crashes; last resort) |

## 8 · Common mistakes

- **Free-text routing.** Always `Literal` + structured output, criteria in the prompt.
- **No default branch.** Low-confidence or unexpected cases need somewhere safe to go.
- **Relying on `recursion_limit` as the loop guard.** It crashes. Count in state and exit gracefully.
- **Forgetting the reducer on `Send` outputs.** Parallel workers writing one key need `operator.add` (or similar).
- **Wrong-sized `Send` payloads.** The whole state is wasteful; the topic alone loses the
  context. Send exactly what the worker needs (here: the question *and* its topic).
- **Unbounded fan-out.** A planner that returns 50 topics will fire 50 calls. Cap it in
  the schema/prompt, and set `max_concurrency`.
- **`Command` without a return annotation** (or `destinations=`). The graph still runs,
  but it can't be drawn or validated properly.

## 9 · Check yourself

<details><summary><b>1.</b> Name the three rules that make an LLM router safe.</summary>

Structured output with a `Literal` of the real branch names; the routing criteria in the prompt (so the model actually reads them); and a code-level default for low confidence or unexpected cases.
</details>

<details><summary><b>2.</b> When would you prefer <code>Command</code> over a conditional edge?</summary>

When the node that does the work is the natural place to decide what comes next, since update and destination are computed together, and for agent handoffs. Prefer conditional edges when the routing logic is separate, reused, or you want all routing visible at the graph level.
</details>

<details><summary><b>3.</b> What two guards does a refine loop need, and why not just rely on <code>recursion_limit</code>?</summary>

A hard cap on iterations (a counter in state checked by the router), and a graceful exit that returns the best result with an honest status. `recursion_limit` is a crash, not an outcome: the caller gets an exception instead of a flagged draft.
</details>

<details><summary><b>4.</b> What's the difference between static fan-out and <code>Send</code>?</summary>

Static fan-out has branches drawn at build time (a fixed number), and every branch reads the full state. `Send` creates branches at runtime, any number decided by data or the model, and each receives its own payload.
</details>

<details><summary><b>5.</b> In the map-reduce graph, why does <code>write</code> run once and see all findings?</summary>

All `Send` branches run in the same superstep, their updates are merged through the `operator.add` reducer on `findings`, and only then does the next superstep (`write`) start.
</details>

<details><summary><b>6.</b> Your planner occasionally returns 30 topics and your local GPU grinds to a halt. Two fixes?</summary>

Cap the number of topics (in the schema description and in code, e.g. `topics[:5]`), and set `max_concurrency` in the run config so only a few workers run at once.
</details>

## Takeaway

LangGraph's control flow is a small toolbox: **edges** for guarantees,
**conditional edges** and **`Command`** for decisions, **cycles with counters**
for refinement, **static fan-out** and **`Send`** for parallelism. Let the model
choose only among branches you drew, guard every loop with state, and bound
every fan-out.

Next → `06_multi_agent.ipynb`: several agents working together, with the
supervisor pattern and handoffs.